# 05 · Panels (`panel_event`, `panel_hazard`)

Inputs: tables from 04, 04b and 04c (`key_flags.is_service` includes the Coinbase migration keys and the 326 keys identified in 04c)

Panel construction only; estimation is in 20.

| Table | Unit | Content |
|---|---|---|
| `panel_keys` | key × event | risk set (balance at t−1 ≥ 0.01 BTC, incoming transfer in the previous 12 months, `NOT is_service`), six CEM bins, `is_large_mover` |
| `panel_key_flows` | key × event × relative day (days with outflows) | amount per destination, cumulative share (capped at 1) and its increment |
| `panel_hazard` | key × event | relative day of first de-risking (none = right-censored at `censor_day`) |
| `panel_event` | CEM cell × group × event × relative day 0…+30 | key- and BTC-weighted daily and cumulative outcomes, sums of squares, life table |
| `panel_cem_strata` | event × CEM cell | keys and BTC in L / C / T, common-support flags, CEM weights |

- Window: relative days 0…+30 (snapshot at the close of t−1, so day 0 is the event day).
- Outcomes: v = outflow to a destination / balance at t−1, capped at 1; daily v_d and cumulative v_c (summed from day 0, capped at 1). Key-weighted (main): cell sum ÷ number of keys. BTC-weighted (secondary): Σ min(balance, cumulative outflow) ÷ Σ balance.
- Destinations: `y1_strict` (main), `y1_mid`, `y1_loose`, `y2` (exchange), `y0` (to an exposed address; negative control), `y3` (any outflow); `y1_strict_held` and `y3_held` count only coins held before t−1 (transactions with `flow_key_tx.newest_in_created ≤ snap_date`; transactions that also spend newer coins are dropped entirely).

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

RISK_MIN_BTC       = 0.01    # risk set: minimum balance at t-1
RISK_LOOKBACK_DAYS = 365     # risk set: incoming transfer within 12 months (not applied to positive controls)
FREEZE_DATE        = "2026-09-22"
POST_MAX           = 30      # relative days 0...+30
REPORT_DAYS        = (7, 30) # cumulative horizons reported

# CEM coarsening (cut points, left-closed, right-open)
BAL_EDGES_BTC = [1, 10, 100];         BAL_LABELS = ["0.01-1", "1-10", "10-100", "100+"]
AGE_EDGES_D   = [365, 730, 1826];     AGE_LABELS = ["<1y", "1-2y", "2-5y", "5y+"]
FS_EDGES_YEAR = [2011, 2016, 2020, 2023]; FS_LABELS = ["<=2010", "2011-15", "2016-19", "2020-22", "2023+"]
REC_EDGES_D   = [30, 180, 365];       REC_LABELS = ["<30d", "30-180d", "180-365d", "365d+"]
NUTXO_EDGES   = [3, 11];              NUTXO_LABELS = ["1-2", "3-10", "11+"]
FAMILY = {"k": "hash160", "s": "p2sh", "w": "p2wsh", "t": "taproot"}

# outcomes (flow_key_tx column -> panel name); *_held = only coins held before t-1
OUTCOMES = ["y1_strict", "y1_mid", "y1_loose", "y2", "y0", "y3", "y1_strict_held", "y3_held"]
HELD = {"y1_strict_held": "y1_strict", "y3_held": "y3"}
# hazard "events": (name, outcome, threshold); threshold 0 means cumulative >= 1 satoshi
HAZARDS = [("y1s_gt0", "y1_strict", 0), ("y1s_10", "y1_strict", 0.1), ("y1s_50", "y1_strict", 0.5),
           ("y1s_90", "y1_strict", 0.9), ("y1sh_50", "y1_strict_held", 0.5),
           ("y3_50", "y3", 0.5), ("y0_50", "y0", 0.5)]

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, re, time, datetime as dt
import numpy as np
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["holder_snapshot", "holder_snapshot_placebo", "holder_snapshot_pc", "key_flags", "flow_days",
          "flow_key_tx", "key_day_flows", "event_flow_coverage", "placebo_draws_strict", "large_movers", "custodial_keys"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from earlier steps found. Spent in this Colab session: $%.2f / $%.0f (the ledger file lives in the current session only)" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def case_bins(expr, edges, labels, null_label="na"):
    # left-closed, right-open: expr < edges[0] -> labels[0] ...; expr NULL -> null_label
    assert len(labels) == len(edges) + 1
    parts = [f"WHEN {expr} IS NULL THEN '{null_label}'"]
    parts += [f"WHEN {expr} < {e} THEN '{l}'" for e, l in zip(edges, labels[:-1])]
    return "CASE " + " ".join(parts) + f" ELSE '{labels[-1]}' END"

SATS_MIN = int(round(RISK_MIN_BTC * 1e8))
DATA_END = pd.Timestamp(FREEZE_DATE)
_latest = pd.Timestamp(q(f"SELECT MAX(snap_date) AS d FROM `{DS}.holder_snapshot` WHERE event = 'latest'").d.iloc[0])
assert _latest == DATA_END, f"latest snapshot date {_latest.date()} differs from the freeze date {FREEZE_DATE}"
print("Freeze date:", DATA_END.date())

## Step 1 · Risk set and CEM bins, `panel_keys`

The three snapshot tables (UNION ALL) joined to `event_flow_coverage` (event type, `strict_ok`, `cb_confounded`) and `key_flags` (first-seen date, `is_service`); `is_service` keys dropped. One row per key × event.

- Events: main and secondary events (`holder_snapshot` except latest), 38 placebo and negative-control dates (`holder_snapshot_placebo`), positive controls FTX and Randstorm (`holder_snapshot_pc`, no recent-inflow requirement; `recent_in` restores the main definition).
- y1-strict censoring: main events and positive controls are fully observed for 30 days; placebo dates carry `strict_ok` and `cb_confounded`, and randomization inference uses `placebo_draws_strict`.
- `is_large_mover`: keys in 04c `large_movers` (≥ 1,000 BTC sent in one day); a separate cell dimension, so BTC-weighted robustness checks can drop these cells.

CEM bins:

| Dimension | Bins | Source |
|---|---|---|
| Balance | 0.01–1, 1–10, 10–100, ≥100 BTC | balance at t−1 |
| Coin age | < 1, 1–2, 2–5, ≥ 5 years | creation date of the oldest unspent output at t−1 |
| Script family | hash160 (P2PK/P2PKH/P2WPKH), P2SH, P2WSH (group T = taproot) | key_kind |
| First seen | ≤ 2010, 2011–15, 2016–19, 2020–22, ≥ 2023 | key_flags.first_seen_date |
| Last incoming transfer | < 30, 30–180, 180–365 days (≥ 365 for positive controls) | last_in_date at t−1 |
| Number of UTXOs | 1–2, 3–10, ≥ 11 | n_utxo at t−1 |

The 2011–15 first-seen bin contains the Randstorm cohort. Transaction history is proxied by UTXOs at t−1 because `key_flags.key_n_received` includes post-event receipts.

Checks: risk-set and excluded service keys per event; keys missing from `key_flags` = 0.

In [ ]:
SNAP_COLS = "event, event_date, snap_date, key_id, key_kind, sats, n_utxo, oldest_utxo_date, last_in_date, grp"
def recent(a=""):
    p = f"{a}." if a else ""
    return f"{p}last_in_date > DATE_SUB({p}snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)"
RECENT = recent()
FAM_SQL = "CASE key_kind " + " ".join(f"WHEN '{k}' THEN '{v}'" for k, v in FAMILY.items()) + " END"
DIMS = ["bal_bin", "age_bin", "family", "fs_bin", "rec_bin", "nutxo_bin"]

SQL_PK = f'''
CREATE OR REPLACE TABLE `{DS}.panel_keys`
CLUSTER BY event, grp
AS
WITH s AS (
  SELECT {SNAP_COLS}, 'main' AS snap_src FROM `{DS}.holder_snapshot`
  WHERE event != 'latest' AND sats >= {SATS_MIN} AND {RECENT}
  UNION ALL
  SELECT {SNAP_COLS}, 'placebo' AS snap_src FROM `{DS}.holder_snapshot_placebo`
  WHERE sats >= {SATS_MIN} AND {RECENT}
  UNION ALL
  SELECT {SNAP_COLS}, 'pc' AS snap_src FROM `{DS}.holder_snapshot_pc`
  WHERE sats >= {SATS_MIN}
),
lm AS (SELECT DISTINCT FROM_HEX(LOWER(key_id_hex)) AS key_id FROM `{DS}.large_movers`),
b AS (
  SELECT s.*,
         c.event_set, c.strict_ok,
         IFNULL(c.cb_confounded, FALSE)                                         AS cb_confounded,
         LEAST({POST_MAX}, DATE_DIFF(DATE '{FREEZE_DATE}', s.event_date, DAY))  AS censor_day,
         {recent("s")}                                                           AS recent_in,
         k.key_id IS NULL                                                       AS no_flags,
         IFNULL(k.is_service, FALSE)                                            AS is_service,
         k.first_seen_date,
         lm.key_id IS NOT NULL                                                  AS is_large_mover
  FROM s
  JOIN `{DS}.event_flow_coverage` c ON c.event = s.event
  LEFT JOIN `{DS}.key_flags` k ON k.key_id = s.key_id
  LEFT JOIN lm ON lm.key_id = s.key_id
),
d AS (
  SELECT b.* EXCEPT(is_service),
    {case_bins("sats", [int(e * 1e8) for e in BAL_EDGES_BTC], BAL_LABELS)}                   AS bal_bin,
    {case_bins("DATE_DIFF(snap_date, oldest_utxo_date, DAY)", AGE_EDGES_D, AGE_LABELS)}      AS age_bin,
    {FAM_SQL}                                                                                AS family,
    {case_bins("EXTRACT(YEAR FROM first_seen_date)", FS_EDGES_YEAR, FS_LABELS)}              AS fs_bin,
    {case_bins("DATE_DIFF(snap_date, last_in_date, DAY)", REC_EDGES_D, REC_LABELS)}          AS rec_bin,
    {case_bins("n_utxo", NUTXO_EDGES, NUTXO_LABELS)}                                         AS nutxo_bin
  FROM b
  WHERE NOT b.is_service
)
SELECT d.*, CONCAT({", '|', ".join(DIMS)}) AS cem_id
FROM d
'''
build("panel_keys", SQL_PK)

pk_summary = q(f'''
SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date, grp,
       COUNT(*) AS n_keys, ROUND(SUM(sats)/1e8) AS btc, COUNTIF(no_flags) AS no_flags,
       COUNTIF(is_large_mover) AS n_large_mover, COUNTIF(recent_in) AS n_recent_in,
       COUNT(DISTINCT cem_id) AS n_strata, ANY_VALUE(censor_day) AS censor_day
FROM `{DS}.panel_keys` GROUP BY event, grp ORDER BY event_date, grp''')
assert pk_summary.no_flags.sum() == 0, "Some risk-set keys are not in key_flags (inconsistent with 04's flow_universe)"
dropped = q(f'''
SELECT h.event, COUNT(*) AS n_service_dropped
FROM `{DS}.holder_snapshot` h JOIN `{DS}.key_flags` k USING (key_id)
WHERE h.event != 'latest' AND h.sats >= {SATS_MIN} AND {recent("h")} AND k.is_service
GROUP BY 1 ORDER BY 1''')
display(pk_summary[pk_summary.event_set != "placebo"])
display(pk_summary[pk_summary.event_set == "placebo"].groupby("grp").n_keys.describe())
display(dropped)

## Step 2 · `panel_key_flows`

`flow_key_tx` aggregated to key × event × relative day 0…+30 via `flow_days`, including the held-coins version. Stored: cumulative share `vc_*` (capped at 1), its increment `dvc_*`, the increment of its square `dvc2_*`, and the first-positive-day indicator `new_any_*`. Summing increments in `panel_event` gives cell sums of cumulative shares without a dense key × day panel.

In [ ]:
need("panel_keys")
fr = q(f"SELECT MIN(date) AS lo, MAX(date) AS hi FROM `{DS}.flow_days` WHERE rel_day BETWEEN 0 AND {POST_MAX}")
FLOW_LO, FLOW_HI = pd.Timestamp(fr.lo.iloc[0]), pd.Timestamp(fr.hi.iloc[0])
BASE = [o for o in OUTCOMES if o not in HELD]

sum_cols = ",\n    ".join([f"SUM(t.{o}) AS {o}" for o in BASE] +
                          [f"SUM(IF(t.newest_in_created <= pk.snap_date, t.{src}, 0)) AS {o}" for o, src in HELD.items()])
cum_cols = ",\n    ".join(f"SUM({o}) OVER w AS cy_{o}" for o in OUTCOMES)
vc_cols  = ",\n    ".join(f"LEAST(1.0, cy_{o} / bal) AS vc_{o}" for o in OUTCOMES)
out_cols = ",\n  ".join(
    f"{o}, vc_{o},\n  vc_{o} - IFNULL(LAG(vc_{o}) OVER w2, 0) AS dvc_{o},\n"
    f"  vc_{o} * vc_{o} - POW(IFNULL(LAG(vc_{o}) OVER w2, 0), 2) AS dvc2_{o},\n"
    f"  IF(cy_{o} >= 1 AND IFNULL(LAG(cy_{o}) OVER w2, 0) < 1, 1, 0) AS new_any_{o}"
    for o in OUTCOMES)

SQL_PKF = f'''
CREATE OR REPLACE TABLE `{DS}.panel_key_flows`
CLUSTER BY event, grp
AS
WITH fd AS (
  SELECT event, rel_day, date FROM `{DS}.flow_days` WHERE rel_day BETWEEN 0 AND {POST_MAX}
),
t AS (
  SELECT fd.event, fd.rel_day, f.key_id, f.newest_in_created,
         {", ".join(f"f.{o}" for o in BASE)}
  FROM `{DS}.flow_key_tx` f
  JOIN fd ON fd.date = f.date
  WHERE f.date BETWEEN DATE '{FLOW_LO:%Y-%m-%d}' AND DATE '{FLOW_HI:%Y-%m-%d}'
),
a AS (
  SELECT t.event, t.key_id, t.rel_day,
    ANY_VALUE(pk.sats) AS bal, ANY_VALUE(pk.grp) AS grp, ANY_VALUE(pk.cem_id) AS cem_id,
    ANY_VALUE(pk.is_large_mover) AS is_large_mover,
    {sum_cols}
  FROM t
  JOIN `{DS}.panel_keys` pk ON pk.event = t.event AND pk.key_id = t.key_id
  GROUP BY t.event, t.key_id, t.rel_day
),
c AS (
  SELECT a.*,
    {cum_cols}
  FROM a
  WINDOW w AS (PARTITION BY event, key_id ORDER BY rel_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
),
v AS (
  SELECT c.*,
    {vc_cols}
  FROM c
)
SELECT event, key_id, rel_day, bal, grp, cem_id, is_large_mover,
  {out_cols}
FROM v
WINDOW w2 AS (PARTITION BY event, key_id ORDER BY rel_day)
'''
build("panel_key_flows", SQL_PKF)

pkf_chk = q(f'''
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT CONCAT(event, TO_HEX(key_id))) AS n_key_events,
  COUNTIF(vc_y3 < -1e-9 OR vc_y3 > 1 + 1e-9) AS bad_range,
  COUNTIF(dvc_y1_strict < -1e-9 OR dvc_y3 < -1e-9) AS neg_increment,
  COUNTIF(y1_strict_held > y1_strict + 1 OR y3_held > y3 + 1) AS held_gt_all,
  COUNTIF(y1_strict > y1_loose + 1 OR y1_mid > y1_loose + 1) AS strict_gt_loose,
  ROUND(SUM(y3)/1e8) AS y3_btc, ROUND(SUM(y1_strict)/1e8) AS y1_strict_btc
FROM `{DS}.panel_key_flows`''')
display(pkf_chk)
# reconcile totals with key_day_flows
kdf_tot = q(f'''
SELECT ROUND(SUM(f.y3)/1e8) AS y3_btc, ROUND(SUM(f.y1_strict)/1e8) AS y1_strict_btc
FROM `{DS}.panel_keys` pk
JOIN `{DS}.flow_days` d ON d.event = pk.event AND d.rel_day BETWEEN 0 AND {POST_MAX}
JOIN `{DS}.key_day_flows` f ON f.key_id = pk.key_id AND f.date = d.date
WHERE f.date BETWEEN DATE '{FLOW_LO:%Y-%m-%d}' AND DATE '{FLOW_HI:%Y-%m-%d}' ''')
display(kdf_tot)
c = pkf_chk.iloc[0]
assert c.bad_range == 0 and c.neg_increment == 0 and c.held_gt_all == 0 and c.strict_gt_loose == 0
assert abs(c.y3_btc - kdf_tot.y3_btc.iloc[0]) <= 1 and abs(c.y1_strict_btc - kdf_tot.y1_strict_btc.iloc[0]) <= 1
print("panel_key_flows checks passed")

## Step 3 · `panel_hazard`

One row per risk-set key × event: relative day `t_*` on which each threshold is first reached (NULL = right-censored at `censor_day` = min(+30, freeze date − event date)), with CEM bins, balance and first-seen date, for Cox, piecewise-exponential and Kaplan–Meier models in 20. Main definition `t_y1s_50`: cumulative y1 strict ≥ 50% of the balance at t−1; also > 0, ≥ 10%, ≥ 90%, held coins ≥ 50%, y3 ≥ 50%, y0 ≥ 50%.

In [ ]:
need("panel_key_flows")
def hz_cond(src, thr):
    return f"vc_{src} * bal >= 1" if thr == 0 else f"vc_{src} >= {thr}"
hz_cols = ",\n    ".join(f"MIN(IF({hz_cond(src, thr)}, rel_day, NULL)) AS t_{name}" for name, src, thr in HAZARDS)
SQL_HZ = f'''
CREATE OR REPLACE TABLE `{DS}.panel_hazard`
CLUSTER BY event, grp
AS
WITH h AS (
  SELECT event, key_id,
    {hz_cols}
  FROM `{DS}.panel_key_flows`
  GROUP BY event, key_id
)
SELECT pk.event, pk.event_set, pk.event_date, pk.snap_date, pk.key_id, pk.grp, pk.cem_id,
       {", ".join(f"pk.{d}" for d in DIMS)},
       pk.is_large_mover, pk.recent_in, pk.first_seen_date, pk.sats,
       pk.censor_day, pk.strict_ok, pk.cb_confounded,
       {", ".join(f"h.t_{name}" for name, _, _ in HAZARDS)}
FROM `{DS}.panel_keys` pk
LEFT JOIN h ON h.event = pk.event AND h.key_id = pk.key_id
'''
build("panel_hazard", SQL_HZ)
hz_chk = q(f'''
SELECT COUNT(*) AS n_rows, {", ".join(f"COUNTIF(t_{n} IS NOT NULL) AS n_{n}" for n, _, _ in HAZARDS)},
       COUNTIF(t_y1s_50 > censor_day) AS t_after_censor,
       COUNTIF(t_y1s_10 > t_y1s_50 OR t_y1s_gt0 > t_y1s_10 OR t_y1s_50 > t_y1s_90) AS non_monotone
FROM `{DS}.panel_hazard`''')
display(hz_chk)
assert hz_chk.t_after_censor.iloc[0] == 0 and hz_chk.non_monotone.iloc[0] == 0
assert hz_chk.n_rows.iloc[0] == pk_summary.n_keys.sum()

## Step 4 · `panel_event`

Cell = event × group (L/T/C) × CEM cell × `is_large_mover`, expanded over relative days 0…`censor_day`. Seven columns per outcome `o`:

| Column | Meaning | Cell mean |
|---|---|---|
| `o_d`, `o_d2` | Σ daily share min(1, y/balance), and its sum of squares | key-weighted daily = `o_d / n_keys` |
| `o_db` | Σ min(balance, daily y) (BTC) | BTC-weighted daily = `o_db / bal_btc` |
| `o_c`, `o_c2` | Σ cumulative share v_c, and its sum of squares | key-weighted cumulative = `o_c / n_keys` (main outcome) |
| `o_cb` | Σ min(balance, cumulative y) (BTC) | BTC-weighted cumulative = `o_cb / bal_btc` |
| `o_any` | keys with cumulative ≥ 1 satoshi | extensive margin |

Life table per hazard event `h`: `h_n` (keys first reaching the threshold that day) and `h_risk` (keys at risk at the start of the day). Sums of squares give within-cell variances for 20.

In [ ]:
need("panel_keys", "panel_key_flows", "panel_hazard")
KEYS = ["event", "grp", "cem_id", "is_large_mover"]
KJ = ", ".join(KEYS)
fl_cols = ",\n    ".join(
    f"SUM(LEAST(1.0, {o} / bal)) AS {o}_d, SUM(POW(LEAST(1.0, {o} / bal), 2)) AS {o}_d2, "
    f"SUM(LEAST(bal, {o})) / 1e8 AS {o}_db,\n    "
    f"SUM(dvc_{o}) AS {o}_ic, SUM(dvc2_{o}) AS {o}_ic2, SUM(bal * dvc_{o}) / 1e8 AS {o}_icb, SUM(new_any_{o}) AS {o}_ina"
    for o in OUTCOMES)
hz_unnest = ", ".join(f"STRUCT('{n}' AS o, h.t_{n} AS t)" for n, _, _ in HAZARDS)
hz_cnt = ",\n    ".join(f"COUNTIF(x.o = '{n}') AS h_{n}_n" for n, _, _ in HAZARDS)
W  = "PARTITION BY event, grp, cem_id, is_large_mover ORDER BY rel_day"
out = []
for o in OUTCOMES:
    out += [f"IFNULL({o}_d, 0) AS {o}_d", f"IFNULL({o}_d2, 0) AS {o}_d2", f"IFNULL({o}_db, 0) AS {o}_db",
            f"SUM(IFNULL({o}_ic, 0)) OVER w AS {o}_c", f"SUM(IFNULL({o}_ic2, 0)) OVER w AS {o}_c2",
            f"SUM(IFNULL({o}_icb, 0)) OVER w AS {o}_cb", f"SUM(IFNULL({o}_ina, 0)) OVER w AS {o}_any"]
for n, _, _ in HAZARDS:
    out += [f"IFNULL(h_{n}_n, 0) AS h_{n}_n",
            f"n_keys - IFNULL(SUM(IFNULL(h_{n}_n, 0)) OVER wp, 0) AS h_{n}_risk"]
out_sql = ",\n  ".join(out)

SQL_PE = f'''
CREATE OR REPLACE TABLE `{DS}.panel_event`
CLUSTER BY event, grp
AS
WITH cells AS (
  SELECT {KJ},
         ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date,
         {", ".join(f"ANY_VALUE({d}) AS {d}" for d in DIMS)},
         ANY_VALUE(censor_day) AS censor_day, ANY_VALUE(strict_ok) AS strict_ok,
         ANY_VALUE(cb_confounded) AS cb_confounded,
         COUNT(*) AS n_keys, COUNTIF(recent_in) AS n_recent_in, SUM(sats) / 1e8 AS bal_btc
  FROM `{DS}.panel_keys`
  GROUP BY {KJ}
),
fl AS (
  SELECT {KJ}, rel_day,
    {fl_cols}
  FROM `{DS}.panel_key_flows`
  GROUP BY {KJ}, rel_day
),
hz AS (
  SELECT {", ".join(f"h.{k}" for k in KEYS)}, x.t AS rel_day,
    {hz_cnt}
  FROM `{DS}.panel_hazard` h, UNNEST([{hz_unnest}]) AS x
  WHERE x.t IS NOT NULL
  GROUP BY {", ".join(f"h.{k}" for k in KEYS)}, x.t
),
d AS (
  SELECT cells.*, r AS rel_day
  FROM cells, UNNEST(GENERATE_ARRAY(0, {POST_MAX})) AS r
  WHERE r <= cells.censor_day
),
j AS (
  SELECT d.*, fl.* EXCEPT({KJ}, rel_day), hz.* EXCEPT({KJ}, rel_day)
  FROM d
  LEFT JOIN fl USING ({KJ}, rel_day)
  LEFT JOIN hz USING ({KJ}, rel_day)
)
SELECT {KJ}, rel_day, event_set, event_date,
  {", ".join(DIMS)},
  censor_day, strict_ok, cb_confounded, n_keys, n_recent_in, bal_btc,
  {out_sql}
FROM j
WINDOW w AS ({W} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
       wp AS ({W} ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
'''
build("panel_event", SQL_PE)

### Checks on `panel_event`

Key counts against `panel_keys`, outcome sums against `panel_key_flows`, non-decreasing cumulative columns, life-table identities.

In [ ]:
need("panel_event")
pe_chk = q(f'''
WITH a AS (
  SELECT
    (SELECT SUM(n_keys) FROM `{DS}.panel_event` WHERE rel_day = 0)                       AS n_keys_pe,
    (SELECT COUNT(*) FROM `{DS}.panel_keys`)                                            AS n_keys_pk,
    (SELECT SUM(y1_strict_d) FROM `{DS}.panel_event`)                                   AS d_pe,
    (SELECT SUM(LEAST(1.0, y1_strict / bal)) FROM `{DS}.panel_key_flows`)               AS d_pkf,
    (SELECT SUM(y3_c) FROM `{DS}.panel_event` WHERE rel_day = 30)                       AS c_pe,
    (SELECT SUM(vc) FROM (SELECT MAX(f.vc_y3) AS vc FROM `{DS}.panel_key_flows` f
                          JOIN `{DS}.panel_keys` k ON k.event = f.event AND k.key_id = f.key_id
                          WHERE k.censor_day = 30 GROUP BY f.event, f.key_id))          AS c_pkf,
    (SELECT SUM(h_y1s_50_n) FROM `{DS}.panel_event`)                                    AS h_pe,
    (SELECT COUNTIF(t_y1s_50 IS NOT NULL) FROM `{DS}.panel_hazard`)                     AS h_hz
),
b AS (
  SELECT
    COUNTIF(y1_strict_c > n_keys + 1e-6 OR y3_any > n_keys)                             AS c_gt_n,
    COUNTIF(rel_day = 0 AND h_y1s_50_risk != n_keys)                                    AS risk0_bad,
    COUNTIF(h_y1s_50_risk - h_y1s_50_n < 0)                                             AS risk_neg
  FROM `{DS}.panel_event`
),
m AS (
  SELECT COUNTIF(y3_c < prev - 1e-9) AS non_monotone
  FROM (SELECT y3_c, LAG(y3_c) OVER (PARTITION BY event, grp, cem_id, is_large_mover ORDER BY rel_day) AS prev
        FROM `{DS}.panel_event`)
  WHERE prev IS NOT NULL
),
r AS (
  SELECT COUNTIF(n != censor_day + 1) AS bad_rows
  FROM (SELECT ANY_VALUE(censor_day) AS censor_day, COUNT(*) AS n FROM `{DS}.panel_event`
        GROUP BY event, grp, cem_id, is_large_mover)
)
SELECT * FROM a, b, m, r''')
display(pe_chk.T)
c = pe_chk.iloc[0]
assert c.n_keys_pe == c.n_keys_pk
assert abs(c.d_pe - c.d_pkf) < 1e-6 * max(1, c.d_pkf) and abs(c.c_pe - c.c_pkf) < 1e-6 * max(1, c.c_pkf)
assert c.h_pe == c.h_hz and c.c_gt_n == 0 and c.risk0_bad == 0 and c.risk_neg == 0
assert c.non_monotone == 0 and c.bad_rows == 0
print("panel_event checks passed:", table_info("panel_event"))

## Step 5 · CEM common support and weights, `panel_cem_strata`

Per event × CEM cell: keys and BTC in L, C and T; `common_LC`, `common_TC`. CEM weights (Iacus, King and Porro): L weight 1; C in common-support cell s gets `w_C = (n_L,s / n_C,s) × (M_C / M_L)`, with M = keys in the event's common support. T vs C uses the five-dimensional cell `cem_id_nf` without script family (`common_TC`, `w_C_TC`), since T is taproot only. BTC-weighted versions use BTC-based weights (`wb_C`). `is_large_mover` is not a matching dimension.

In [ ]:
need("panel_keys")
NF = [d for d in DIMS if d != "family"]      # group T is taproot only, so T vs C is not stratified by script family
SQL_CEM = f'''
CREATE OR REPLACE TABLE `{DS}.panel_cem_strata`
CLUSTER BY event
AS
WITH s AS (
  SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date, cem_id,
         {", ".join(f"ANY_VALUE({d}) AS {d}" for d in DIMS)},
         CONCAT({", '|', ".join(f"ANY_VALUE({d})" for d in NF)}) AS cem_id_nf,
         COUNTIF(grp = 'L') AS n_L, COUNTIF(grp = 'C') AS n_C, COUNTIF(grp = 'T') AS n_T,
         SUM(IF(grp = 'L', sats, 0)) / 1e8 AS btc_L, SUM(IF(grp = 'C', sats, 0)) / 1e8 AS btc_C,
         SUM(IF(grp = 'T', sats, 0)) / 1e8 AS btc_T
  FROM `{DS}.panel_keys`
  GROUP BY event, cem_id
),
m AS (
  SELECT s.*,
    SUM(n_T) OVER (PARTITION BY event, cem_id_nf) AS n_T_nf,
    SUM(n_C) OVER (PARTITION BY event, cem_id_nf) AS n_C_nf
  FROM s
),
m2 AS (
  SELECT m.*, n_L > 0 AND n_C > 0 AS common_LC, n_T_nf > 0 AND n_C_nf > 0 AS common_TC
  FROM m
),
tot AS (
  SELECT m2.*,
    SUM(IF(common_LC, n_L, 0))   OVER (PARTITION BY event) AS M_L,
    SUM(IF(common_LC, n_C, 0))   OVER (PARTITION BY event) AS M_C,
    SUM(IF(common_LC, btc_L, 0)) OVER (PARTITION BY event) AS MB_L,
    SUM(IF(common_LC, btc_C, 0)) OVER (PARTITION BY event) AS MB_C,
    SUM(IF(common_TC, n_T, 0))   OVER (PARTITION BY event) AS M_T,
    SUM(IF(common_TC, n_C, 0))   OVER (PARTITION BY event) AS M_C_T
  FROM m2
)
SELECT tot.*,
  IF(common_LC, SAFE_DIVIDE(n_L, n_C) * SAFE_DIVIDE(M_C, M_L), 0)            AS w_C,
  IF(common_LC, SAFE_DIVIDE(btc_L, btc_C) * SAFE_DIVIDE(MB_C, MB_L), 0)      AS wb_C,
  IF(common_TC, SAFE_DIVIDE(n_T_nf, n_C_nf) * SAFE_DIVIDE(M_C_T, M_T), 0)    AS w_C_TC   -- on cem_id_nf cells
FROM tot
'''
build("panel_cem_strata", SQL_CEM, force=True)
cem = q(f"SELECT * FROM `{DS}.panel_cem_strata`")
_w = cem[cem.common_LC].assign(wn=lambda d: d.w_C * d.n_C).groupby("event").agg(wn=("wn", "sum"), M_C=("M_C", "first"))
assert np.allclose(_w.wn, _w.M_C), "CEM weight check failed: sum  w_C-n_C != M_C"
_t = cem[cem.common_TC].assign(wn=lambda d: d.w_C_TC * d.n_C).groupby("event").agg(wn=("wn", "sum"), M=("M_C_T", "first"))
assert np.allclose(_t.wn, _t.M), "T vs C weight check failed"
cem_sum = (cem.groupby(["event", "event_set", "event_date"])
              .apply(lambda g: pd.Series({
                  "n_strata": len(g), "n_common_LC": int(g.common_LC.sum()),
                  "L_keys_matched": g.loc[g.common_LC, "n_L"].sum() / max(g.n_L.sum(), 1),
                  "C_keys_matched": g.loc[g.common_LC, "n_C"].sum() / max(g.n_C.sum(), 1),
                  "L_btc_matched": g.loc[g.common_LC, "btc_L"].sum() / max(g.btc_L.sum(), 1e-9),
                  "T_keys_matched": g.loc[g.common_TC, "n_T"].sum() / max(g.n_T.sum(), 1)}))
              .reset_index().sort_values("event_date"))
display(cem_sum[cem_sum.event_set != "placebo"].round(3))
display(cem_sum[cem_sum.event_set == "placebo"][["L_keys_matched", "C_keys_matched", "L_btc_matched", "T_keys_matched"]].describe().round(3))
cem.to_csv("panel_cem_strata.csv", index=False)
cem_sum.to_csv("panel_cem_summary.csv", index=False)

## Step 6 · Descriptive summary

Within common support, C reweighted to the cell distribution of L: cumulative means at +7 and +30 and L − C, key- and BTC-weighted, for y1 strict, y1 strict (held), y3 and y0. Placebo dates from the main pool of `placebo_draws_strict` only. Estimation and inference are in 20. Files: `panel_summary_05.csv`, `fig_panel_05.png`.

In [ ]:
need("panel_event", "panel_cem_strata")
SUM_OUT = ["y1_strict", "y1_strict_held", "y3", "y0"]
agg = q(f'''
SELECT p.event, p.event_set, p.event_date, p.grp, p.cem_id, p.rel_day,
       SUM(p.n_keys) AS n, SUM(p.bal_btc) AS bal,
       {", ".join(f"SUM(p.{o}_c) AS {o}_c, SUM(p.{o}_cb) AS {o}_cb" for o in SUM_OUT)}
FROM `{DS}.panel_event` p
WHERE p.rel_day IN ({", ".join(map(str, REPORT_DAYS))}) AND p.grp IN ('L', 'C')
GROUP BY 1, 2, 3, 4, 5, 6''')

def cem_diff(g):
    L = g[g.grp == "L"].set_index("cem_id"); C = g[g.grp == "C"].set_index("cem_id")
    s = L.index.intersection(C.index)
    L, C = L.loc[s], C.loc[s]
    if len(s) == 0 or L.n.sum() == 0:
        return pd.Series(dtype=float)
    wk, wb = L.n / L.n.sum(), L.bal / L.bal.sum()
    r = {"n_L": L.n.sum(), "n_C": C.n.sum(), "btc_L": L.bal.sum(), "btc_C": C.bal.sum()}
    for o in SUM_OUT:
        r[f"{o}_L"]  = L[f"{o}_c"].sum() / L.n.sum()
        r[f"{o}_C"]  = (wk * C[f"{o}_c"] / C.n).sum()
        r[f"{o}_diff"] = r[f"{o}_L"] - r[f"{o}_C"]
        r[f"{o}_Lb"] = L[f"{o}_cb"].sum() / L.bal.sum()
        r[f"{o}_Cb"] = (wb * C[f"{o}_cb"] / C.bal).sum()
        r[f"{o}_diffb"] = r[f"{o}_Lb"] - r[f"{o}_Cb"]
    return pd.Series(r)

summ = (agg.groupby(["event", "event_set", "event_date", "rel_day"]).apply(cem_diff)
           .reset_index().sort_values(["event_date", "rel_day"]))
summ.to_csv("panel_summary_05.csv", index=False)
show = [c for c in summ.columns if c.endswith(("_L", "_C", "_diff", "_diffb")) and c.startswith("y1_strict_")] + ["n_L", "n_C"]
display(summ[~summ.event_set.isin(["placebo"])].set_index(["event", "rel_day"])[show].round(5))

pool = set(pd.to_datetime(q(f"SELECT DISTINCT fake_date FROM `{DS}.placebo_draws_strict` WHERE pool = 'main'").fake_date))
summ["event_date"] = pd.to_datetime(summ.event_date)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for i, rd in enumerate(REPORT_DAYS):
    for j, (col, lab) in enumerate([("y1_strict_diff", "key-weighted"), ("y1_strict_diffb", "BTC-weighted")]):
        ax = axes[i, j]; s = summ[summ.rel_day == rd]
        pl = s[(s.event_set == "placebo") & s.event_date.isin(pool)]
        mn = s[s.event_set == "main"]
        ax.scatter(pl.event_date, pl[col], s=14, color="grey", label="placebo (strict pool)")
        ax.scatter(mn.event_date, mn[col], s=40, color="C3", label="main events")
        for r in mn.itertuples():
            ax.annotate(r.event, (r.event_date, getattr(r, col)), fontsize=7, xytext=(3, 3), textcoords="offset points")
        ax.axhline(0, color="k", lw=0.6)
        ax.set_title(f"CEM L − C, cumulative y1 strict share, day +{rd} ({lab})", fontsize=9)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig_panel_05.png", dpi=130); plt.show()

In [ ]:
for rec in JOBS[-6:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
TABLES_05 = ["panel_keys", "panel_key_flows", "panel_hazard", "panel_event", "panel_cem_strata"]
for t in TABLES_05:
    n = table_rows(t)
    print(f"{t:20s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "config": {"risk_min_btc": RISK_MIN_BTC, "risk_lookback_days": RISK_LOOKBACK_DAYS,
                     "freeze_date": FREEZE_DATE, "post_max": POST_MAX, "report_days": REPORT_DAYS,
                     "bal_edges_btc": BAL_EDGES_BTC, "age_edges_d": AGE_EDGES_D, "fs_edges_year": FS_EDGES_YEAR,
                     "rec_edges_d": REC_EDGES_D, "nutxo_edges": NUTXO_EDGES, "outcomes": OUTCOMES,
                     "hazards": HAZARDS},
          "ledger": LEDGER,
          "tables": [table_info(t) for t in TABLES_05 if table_rows(t)]}
for k, v in [("panel_keys", "pk_summary"), ("panel_key_flows_check", "pkf_chk"), ("hazard_check", "hz_chk"),
             ("panel_event_check", "pe_chk"), ("cem_summary", "cem_sum")]:
    try: report[k] = eval(v).astype(str).to_dict(orient="records")
    except NameError: pass
with open("build_report_05.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    for fn in ["build_report_05.json", "panel_cem_strata.csv", "panel_cem_summary.csv",
               "panel_summary_05.csv", "fig_panel_05.png"]:
        if os.path.exists(fn): files.download(fn)
except Exception:
    print("Report saved to build_report_05.json")